# MiniGPT: train a tiny causal transformer

In Colab, PyTorch is normally installed. Locally install it using the official instructions linked in the student guide. Run this cell on CPU; a GPU is not required. This notebook uses single-head causal attention and a synthetic corpus. Validation and test are contiguous partitions of the same repeating source, so they are not independent samples of new text.

Run the cells in order. Change values and rerun to explore.

In [ ]:
import math, random
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42); random.seed(42)
# Synthetic text; repeating it makes the tiny demo fast, but limits evaluation.
text = ('the moon is bright. the sun is warm. a cat can nap. a dog can run.\n' * 60)
vocab = sorted(set(text)); stoi = {ch:i for i,ch in enumerate(vocab)}
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
n = len(data); train = data[:int(.8*n)]; val = data[int(.8*n):int(.9*n)]; test = data[int(.9*n):]
block = 16; batch_size = 16
assert min(map(len, (train,val,test))) > block

def batch(split):
    ix = torch.randint(len(split)-block, (batch_size,))
    return torch.stack([split[i:i+block] for i in ix]), torch.stack([split[i+1:i+block+1] for i in ix])

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__(); d=32
        self.token=nn.Embedding(len(vocab),d); self.position=nn.Embedding(block,d)
        self.q=nn.Linear(d,d,bias=False); self.k=nn.Linear(d,d,bias=False); self.v=nn.Linear(d,d,bias=False)
        self.proj=nn.Linear(d,d); self.norm1=nn.LayerNorm(d); self.norm2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.ReLU(),nn.Linear(4*d,d)); self.head=nn.Linear(d,len(vocab))
    def forward(self, idx, targets=None):
        B,T=idx.shape; x=self.token(idx)+self.position(torch.arange(T,device=idx.device))
        z=self.norm1(x); scores=self.q(z) @ self.k(z).transpose(-2,-1) / math.sqrt(z.shape[-1])
        mask=torch.tril(torch.ones(T,T,device=idx.device,dtype=torch.bool))
        weights=F.softmax(scores.masked_fill(~mask,float('-inf')),dim=-1)
        x=x+self.proj(weights @ self.v(z)); x=x+self.ff(self.norm2(x)); logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,len(vocab)),targets.reshape(-1))
        return logits,loss

model=MiniGPT(); opt=torch.optim.AdamW(model.parameters(),lr=0.003)
for step in range(150):
    x,y=batch(train); _,loss=model(x,y); opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0: print('step',step,'training batch loss',round(loss.item(),3))
@torch.no_grad()
def estimate(split):
    model.eval(); losses=[]
    for _ in range(20):
        x,y=batch(split); losses.append(model(x,y)[1].item())
    model.train(); return sum(losses)/len(losses)
print('uniform baseline cross-entropy:',round(math.log(len(vocab)),3))
print('validation loss:',round(estimate(val),3))
@torch.no_grad()
def generate(prompt='the ', count=100):
    model.eval(); ids=[stoi[c] for c in prompt]
    for _ in range(count):
        logits,_=model(torch.tensor([ids[-block:]],dtype=torch.long))
        ids.append(torch.multinomial(F.softmax(logits[0,-1],dim=-1),1).item())
    return ''.join(vocab[i] for i in ids)
print('sample:',generate())

## Your turn

Change one setting, rerun from a fresh kernel, and compare validation loss and samples. After selecting settings, evaluate once using `print(estimate(test))`. Document that this tiny synthetic split has limited generalization value.

Record your prediction, result, and explanation in a new text cell.